# PS04 – Hex Game AI: Monte Carlo Tree Search (MCTS) with UCT

**BITS Pilani WILP | MTech AI/ML | S2 2025-2026**
**Course: AIMLCZG557 / AECLZG557 | Assignment 2 | Weightage: 13%**
**Deadline: 17 Aug 2026, 11:55 PM IST**

---

## Problem Overview

| Item | Detail |
|------|--------|
| Board | N×N grid, 7 ≤ N ≤ 11 |
| Player A (AI, `1`) | Connects **top row** → **bottom row** |
| Player B (Human, `2`) | Connects **left col** → **right col** |
| Adjacency | 6-neighbours: 4 orthogonal + top-right(r-1,c+1) + bottom-left(r+1,c-1) |
| Algorithm | **MCTS** with **UCT**: Selection → Expansion → Simulation → Backpropagation |

---

## Notebook Structure
1. Imports
2. HexBoard – Board Representation & Win Detection
3. MCTSNode – Tree Node + UCT
4. MCTS Algorithm – 4 Phases
5. Output Formatting
6. Input Parser & Human Move Input
7. Game Loop (Interactive + Automated)
8. Test Case 1 – 7×7 Empty Board
9. Test Case 2 – 9×9 Mid-Game Board
10. Run Full Pipeline via inputPS04.txt


## 1. Imports

In [ ]:
import math
import time
import random
import sys
from collections import deque

print("All imports loaded.")


## 2. HexBoard – Board Representation & Win Detection

Win detection uses **BFS** from the starting edge to the goal edge.

Valid 6-adjacency on the square grid:
| Direction | Delta |
|-----------|-------|
| Top | (-1, 0) |
| Bottom | (+1, 0) |
| Left | (0, -1) |
| Right | (0, +1) |
| Top-Right ✓ | (-1, +1) |
| Bottom-Left ✓ | (+1, -1) |
| Top-Left ✗ | (-1, -1) — **FORBIDDEN** |
| Bottom-Right ✗ | (+1, +1) — **FORBIDDEN** |


In [ ]:
class HexBoard:
    """
    N×N Hex game board.
    Player 1 (A - AI)   : connects row 0 to row N-1 (top → bottom)
    Player 2 (B - Human): connects col 0 to col N-1 (left → right)

    6-adjacency (hexagonal on square grid):
        Allowed  : (-1,0),(+1,0),(0,-1),(0,+1),(-1,+1),(+1,-1)
        Forbidden: (-1,-1),(+1,+1)
    """

    _ADJ = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, 1), (1, -1)]

    def __init__(self, n: int):
        if not (7 <= n <= 11):
            print(f"[WARNING] N={n} outside expected range 7-11.")
        self.n = n
        self.board = [[0] * n for _ in range(n)]

    def copy(self):
        obj = object.__new__(HexBoard)
        obj.n = self.n
        obj.board = [row[:] for row in self.board]
        return obj

    def get_neighbours(self, r, c):
        n = self.n
        return [(r+dr, c+dc) for dr, dc in self._ADJ
                if 0 <= r+dr < n and 0 <= c+dc < n]

    def get_empty_cells(self):
        return [(r, c) for r in range(self.n) for c in range(self.n)
                if self.board[r][c] == 0]

    def is_valid_move(self, r, c):
        return 0 <= r < self.n and 0 <= c < self.n and self.board[r][c] == 0

    def apply_move(self, r, c, player):
        if not (0 <= r < self.n and 0 <= c < self.n):
            raise ValueError(f"({r},{c}) out of bounds.")
        if self.board[r][c] != 0:
            raise ValueError(f"({r},{c}) already occupied by {self.board[r][c]}.")
        self.board[r][c] = player

    # ── Win detection (BFS) ──────────────────────────────────────────

    def check_winner(self):
        """Returns 1, 2, or 0."""
        src_a = [(0, c) for c in range(self.n) if self.board[0][c] == 1]
        if src_a and self._bfs(1, src_a, goal_r=self.n - 1):
            return 1
        src_b = [(r, 0) for r in range(self.n) if self.board[r][0] == 2]
        if src_b and self._bfs(2, src_b, goal_c=self.n - 1):
            return 2
        return 0

    def _bfs(self, player, starts, goal_r=None, goal_c=None):
        visited = set(starts)
        queue = deque(starts)
        while queue:
            r, c = queue.popleft()
            if (goal_r is not None and r == goal_r) or                (goal_c is not None and c == goal_c):
                return True
            for nr, nc in self.get_neighbours(r, c):
                if (nr, nc) not in visited and self.board[nr][nc] == player:
                    visited.add((nr, nc))
                    queue.append((nr, nc))
        return False

    def get_winning_path(self, player):
        """BFS with parent tracking; returns ordered path or []."""
        if player == 1:
            starts = [(0, c) for c in range(self.n) if self.board[0][c] == 1]
            is_goal = lambda r, _c: r == self.n - 1
        else:
            starts = [(r, 0) for r in range(self.n) if self.board[r][0] == 2]
            is_goal = lambda _r, c: c == self.n - 1

        parent = {s: None for s in starts}
        queue = deque(starts)
        goal_cell = None
        while queue and goal_cell is None:
            r, c = queue.popleft()
            if is_goal(r, c):
                goal_cell = (r, c)
                break
            for nr, nc in self.get_neighbours(r, c):
                if (nr, nc) not in parent and self.board[nr][nc] == player:
                    parent[(nr, nc)] = (r, c)
                    queue.append((nr, nc))

        if goal_cell is None:
            return []
        path, cur = [], goal_cell
        while cur is not None:
            path.append(cur)
            cur = parent[cur]
        return list(reversed(path))

    def __str__(self):
        sym = {0: '.', 1: '1', 2: '2'}
        hdr = '    ' + '  '.join(str(c) for c in range(self.n))
        rows = []
        for r in range(self.n):
            indent = ' ' * r
            cells = ' '.join(sym[self.board[r][c]] for c in range(self.n))
            rows.append(f"{r:2}  {indent}{cells}")
        return '\n'.join([hdr] + rows)

print("HexBoard defined.")


In [ ]:
# Quick test: 7x7 board with a winning path for Player A (column 2)
b = HexBoard(7)
for r in range(7):
    b.board[r][2] = 1   # vertical stripe -> Player A wins

print(b)
print(f"\nWinner: {b.check_winner()}  (expected 1)")
print(f"Winning path: {b.get_winning_path(1)}")

# Adjacency test for corner and centre cells
b2 = HexBoard(7)
print(f"\nNeighbours of (0,0): {b2.get_neighbours(0,0)}")   # corner – 2 valid
print(f"Neighbours of (3,3): {b2.get_neighbours(3,3)}")     # centre – 6 valid
print(f"Neighbours of (6,6): {b2.get_neighbours(6,6)}")     # corner – 2 valid


## 3. MCTSNode – Tree Node + UCT

**UCT formula** (Kocsis & Szepesvári, 2006):

```
UCT(v') = Q(v') / N(v') + c * sqrt(ln N(v) / N(v'))
```

- **Q/N** = exploitation (win rate)
- **c√(ln N / N')** = exploration bonus  (c = √2 by default)

**Win attribution convention used here:**
- `node.player` = the player **next to move**
- `node.wins` = wins for the player who **moved to reach** this node = `3 - node.player`

This ensures the parent always selects the child that maximises wins for the mover.


In [ ]:
class MCTSNode:
    """
    MCTS tree node.

    Conventions:
        self.player  : next player to move at this state
        self.wins    : wins for the mover TO this node  (= 3 - self.player)
        self.visits  : total rollouts at this node
    """

    __slots__ = ('board', 'player', 'parent', 'children', 'untried_moves',
                 'move', 'depth', 'visits', 'wins')

    def __init__(self, board, player, parent=None, move=None, depth=0):
        self.board = board
        self.player = player
        self.parent = parent
        self.children = []
        self.move = move
        self.depth = depth
        self.visits = 0
        self.wins = 0.0
        self.untried_moves = board.get_empty_cells()
        random.shuffle(self.untried_moves)  # avoid systematic bias

    def uct_value(self, c=math.sqrt(2)):
        """UCT formula; returns +inf for unvisited nodes."""
        if self.visits == 0:
            return float('inf')
        return (self.wins / self.visits) + c * math.sqrt(
            math.log(self.parent.visits) / self.visits
        )

    def best_child_uct(self):
        return max(self.children, key=lambda ch: ch.uct_value())

    def most_visited_child(self):
        return max(self.children, key=lambda ch: ch.visits)

    @property
    def is_fully_expanded(self):
        return len(self.untried_moves) == 0

    @property
    def is_terminal(self):
        return self.board.check_winner() != 0

print("MCTSNode defined.")


## 4. MCTS Algorithm – 4 Required Phases

```
while time_remaining:
    ┌─ Phase 1: SELECTION ──────────────────────────────────┐
    │  Descend tree using UCT until reaching an             │
    │  unexpanded node or a terminal node.                  │
    ├─ Phase 2: EXPANSION ──────────────────────────────────┤
    │  Add one new child by trying an untried move.         │
    ├─ Phase 3: SIMULATION (ROLLOUT) ───────────────────────┤
    │  Play random moves from the new node to termination.  │
    ├─ Phase 4: BACKPROPAGATION ────────────────────────────┤
    │  Walk up to root, updating visits and wins.           │
    └───────────────────────────────────────────────────────┘
Final move: most-visited child of root (robust against variance).
```


In [ ]:
def _rollout(board, starting_player):
    """
    Simulation: random play to terminal state.
    Hex always has a winner, so this always returns 1 or 2.
    """
    sim = board.copy()
    cur = starting_player
    while True:
        w = sim.check_winner()
        if w:
            return w
        empty = sim.get_empty_cells()
        if not empty:
            return 0   # unreachable in legal Hex
        r, c = random.choice(empty)
        sim.apply_move(r, c, cur)
        cur = 3 - cur


def _backpropagate(node, result):
    """
    Walk from node to root, updating wins/visits.
    node.wins counts wins for (3 - node.player) = mover to this node.
    """
    cur = node
    while cur is not None:
        cur.visits += 1
        if result == (3 - cur.player):
            cur.wins += 1.0
        cur = cur.parent


def run_mcts(board, current_player, time_limit_ms):
    """
    MCTS with UCT.  Returns:
        (best_move, nodes_expanded, max_depth, heuristic_score)
    heuristic_score = estimated win % for current_player (0-100).
    """
    root = MCTSNode(board.copy(), current_player)
    deadline = time.time() + time_limit_ms / 1000.0
    nodes_expanded = 0
    max_depth = 0

    while time.time() < deadline:

        # ---- Phase 1: SELECTION ------------------------------------
        node = root
        while node.is_fully_expanded and node.children and not node.is_terminal:
            node = node.best_child_uct()

        # ---- Phase 2: EXPANSION ------------------------------------
        if node.untried_moves and not node.is_terminal:
            move = node.untried_moves.pop()
            child_board = node.board.copy()
            child_board.apply_move(move[0], move[1], node.player)
            child = MCTSNode(child_board, player=3 - node.player,
                             parent=node, move=move, depth=node.depth + 1)
            node.children.append(child)
            node = child
            nodes_expanded += 1
            if node.depth > max_depth:
                max_depth = node.depth

        # ---- Phase 3: SIMULATION (ROLLOUT) -------------------------
        result = _rollout(node.board, node.player)

        # ---- Phase 4: BACKPROPAGATION ------------------------------
        _backpropagate(node, result)

    if not root.children:
        empty = board.get_empty_cells()
        return (random.choice(empty) if empty else None), 0, 0, 0.0

    best = root.most_visited_child()
    score = (best.wins / best.visits * 100.0) if best.visits else 0.0
    return best.move, nodes_expanded, max_depth, score

print("MCTS algorithm functions defined (rollout, backpropagate, run_mcts).")


## 5. Output Formatting

Builds the exact turn-block format required by the assignment spec.

In [ ]:
def _banner(turn):
    label = f"Turn {turn}"
    return f"{'='*20}{label}{'='*(25 - len(str(turn)))}"


def format_ai_turn(turn, move, depth, nodes, score, elapsed_ms,
                   status, board, winner=0, path=None):
    score_str = '+INF' if winner == 1 else f"{score:.2f}"
    lines = [
        _banner(turn),
        "Player : A (AI)",
        f"Move Selected : ({move[0]},{move[1]})",
        "Search Algorithm : MCTS with UCT",
        f"Search Depth : {depth}",
        f"Nodes Expanded : {nodes:,}",
        f"Heuristic Score : {score_str}",
        f"Execution Time : {int(elapsed_ms)} ms",
        f"Game Status : {status}",
    ]
    if winner == 1 and path:
        path_str = ' -> '.join(f"({r},{c})" for r, c in path)
        lines += ["Terminal State : YES", "Winner : Player A",
                  f"Winning Path {path_str}"]
    lines += ["Current Board", str(board), ""]
    return '\n'.join(lines)


def format_human_turn(turn, move, elapsed_ms, status, board,
                      winner=0, path=None):
    lines = [
        _banner(turn),
        "Player : B (Human)",
        f"Move Entered : ({move[0]},{move[1]})",
        "Move Validation : VALID",
        f"Execution Time : {int(elapsed_ms)} ms",
        f"Game Status : {status}",
    ]
    if winner == 2 and path:
        path_str = ' -> '.join(f"({r},{c})" for r, c in path)
        lines += ["Terminal State : YES", "Winner : Player B",
                  f"Winning Path {path_str}"]
    lines += ["Current Board", str(board), ""]
    return '\n'.join(lines)


def format_summary(winner, total_turns, ai_turns, human_turns,
                   ai_depths, ai_nodes, ai_times):
    avg  = lambda lst: sum(lst) / len(lst) if lst else 0
    peak = lambda lst: max(lst) if lst else 0
    winner_name = "Player A" if winner == 1 else "Player B"
    result = "PLAYER_A_WINS" if winner == 1 else "PLAYER_B_WINS"
    sep = '=' * 60
    lines = [
        f"{sep} GAME OVER {sep}",
        f"Winner : {winner_name}",
        f"Total Turns : {total_turns}",
        f"Total AI Moves : {ai_turns}",
        f"Total Human Moves : {human_turns}",
        f"Average Search Depth : {avg(ai_depths):.1f}",
        f"Average Nodes Expanded : {int(avg(ai_nodes)):,}",
        f"Average AI Move Time : {int(avg(ai_times))} ms",
        f"Maximum Search Depth : {peak(ai_depths)}",
        f"Maximum Nodes Expanded : {int(peak(ai_nodes)):,}",
        f"Game Result : {result}",
    ]
    return '\n'.join(lines)

print("Output formatting functions defined.")


## 6. Input Parser & Human Move Input

In [ ]:
def parse_input_file(filepath):
    """Parse inputPS04.txt. Returns (n, time_limit_ms, board)."""
    try:
        with open(filepath, 'r') as fh:
            raw = [ln.strip() for ln in fh if ln.strip()]
    except FileNotFoundError:
        print(f"[ERROR] File not found: '{filepath}'")
        return None, None, None

    if len(raw) < 2:
        print("[ERROR] Need at least 2 lines.")
        return None, None, None

    n = int(raw[0])
    time_limit_ms = int(raw[1])
    board = HexBoard(n)

    if len(raw) >= n + 2:
        for r in range(n):
            vals = list(map(int, raw[2 + r].split()))
            if len(vals) != n:
                print(f"[ERROR] Row {r}: expected {n} values, got {len(vals)}.")
                return None, None, None
            for c, v in enumerate(vals):
                if v not in (0, 1, 2):
                    print(f"[ERROR] Invalid value {v} at ({r},{c}).")
                    return None, None, None
                board.board[r][c] = v

    return n, time_limit_ms, board


def get_human_move(board):
    """Interactive prompt for Player B. Re-prompts up to 3 times."""
    for attempt in range(1, 4):
        try:
            raw = input("Player B, Enter your move (row,col): ").strip()
            parts = raw.replace(' ', '').split(',')
            if len(parts) != 2:
                raise ValueError
            r, c = int(parts[0]), int(parts[1])
        except (ValueError, IndexError):
            print(f"  [INVALID] Use row,col (e.g. 3,4)  [attempt {attempt}/3]")
            continue

        if not (0 <= r < board.n and 0 <= c < board.n):
            print(f"  [INVALID] ({r},{c}) out of bounds  [attempt {attempt}/3]")
            continue
        if board.board[r][c] != 0:
            print(f"  [INVALID] Cell ({r},{c}) occupied  [attempt {attempt}/3]")
            continue
        return r, c

    print("[ERROR] 3 failed attempts. Exiting.")
    sys.exit(1)

print("Input parser and human move functions defined.")


## 7. Game Loop

- **`run_game()`** – for interactive terminal play (`input()` based)
- **`run_game_test()`** – automated test with predefined human moves (notebook-friendly)


In [ ]:
def run_game(input_file="inputPS04.txt", output_file="outputPS04.txt"):
    """
    Interactive game. Run from terminal, not inside Jupyter.
    Player A (AI) goes first. Player B enters moves via keyboard.
    """
    n, time_limit_ms, board = parse_input_file(input_file)
    if board is None:
        return

    print(f"\n{'='*56}")
    print(f"  HEX GAME  |  {n}x{n}  |  {time_limit_ms} ms/move")
    print(f"  A (AI): top -> bottom   |   B (Human): left -> right")
    print('='*56)
    print(board)

    output_log, turn, current_player = [], 1, 1
    ai_turns = human_turns = 0
    ai_depths, ai_nodes_list, ai_times = [], [], []

    while True:
        winner = board.check_winner()
        if winner:
            break

        if current_player == 1:
            print(f"  Turn {turn} | AI thinking ...", flush=True)
            t0 = time.time()
            move, nodes, depth, score = run_mcts(board, 1, time_limit_ms)
            elapsed_ms = (time.time() - t0) * 1000.0
            if move is None:
                break
            board.apply_move(move[0], move[1], 1)
            ai_turns += 1
            ai_depths.append(depth)
            ai_nodes_list.append(nodes)
            ai_times.append(elapsed_ms)
            winner = board.check_winner()
            status = "TERMINAL" if winner else "CONTINUE"
            path = board.get_winning_path(1) if winner == 1 else []
            output_log.append(
                format_ai_turn(turn, move, depth, nodes, score,
                               elapsed_ms, status, board, winner, path))
            print(f"    -> ({move[0]},{move[1]})  nodes={nodes:,}  "
                  f"score={score:.1f}%  {int(elapsed_ms)}ms")
            print(board)
        else:
            print(f"  Turn {turn} | Player B")
            print(board)
            t0 = time.time()
            r, c = get_human_move(board)
            elapsed_ms = (time.time() - t0) * 1000.0
            board.apply_move(r, c, 2)
            human_turns += 1
            winner = board.check_winner()
            status = "TERMINAL" if winner else "CONTINUE"
            path = board.get_winning_path(2) if winner == 2 else []
            output_log.append(
                format_human_turn(turn, (r, c), elapsed_ms,
                                  status, board, winner, path))

        if winner:
            break
        current_player = 3 - current_player
        turn += 1

    winner = board.check_winner()
    summary = format_summary(winner, turn, ai_turns, human_turns,
                             ai_depths, ai_nodes_list, ai_times)
    output_log.append(summary)
    print(summary)
    with open(output_file, 'w', encoding='utf-8') as fh:
        fh.write('\n'.join(output_log))
    print(f"[OK] Output -> '{output_file}'")


def run_game_test(n, time_limit_ms, initial_board=None,
                  human_moves=None, output_file="outputPS04_test.txt"):
    """
    Notebook-friendly automated test runner.
    Uses predefined human_moves list instead of keyboard input.
    Falls back to first valid empty cell if the list is exhausted.
    Returns dict with game statistics.
    """
    board = HexBoard(n)
    if initial_board:
        for r in range(n):
            for c in range(n):
                board.board[r][c] = initial_board[r][c]

    human_iter = iter(human_moves or [])
    output_log, turn, current_player = [], 1, 1
    ai_turns = human_turns = 0
    ai_depths, ai_nodes_list, ai_times = [], [], []

    print(f"\n{'='*56}")
    print(f"  TEST  |  {n}x{n}  |  {time_limit_ms} ms/move")
    print('='*56)
    print(board)

    while True:
        winner = board.check_winner()
        if winner:
            break

        if current_player == 1:
            t0 = time.time()
            move, nodes, depth, score = run_mcts(board, 1, time_limit_ms)
            elapsed_ms = (time.time() - t0) * 1000.0
            if move is None:
                break
            board.apply_move(move[0], move[1], 1)
            ai_turns += 1
            ai_depths.append(depth)
            ai_nodes_list.append(nodes)
            ai_times.append(elapsed_ms)
            winner = board.check_winner()
            status = "TERMINAL" if winner else "CONTINUE"
            path = board.get_winning_path(1) if winner == 1 else []
            output_log.append(
                format_ai_turn(turn, move, depth, nodes, score,
                               elapsed_ms, status, board, winner, path))
            print(f"  T{turn:02d} AI   -> ({move[0]},{move[1]})  "
                  f"nodes={nodes:,}  score={score:.1f}%  {int(elapsed_ms)}ms")
        else:
            hm = next(human_iter, None)
            if hm is None or not board.is_valid_move(hm[0], hm[1]):
                empty = board.get_empty_cells()
                hm = empty[0] if empty else None
            if hm is None:
                break
            t0 = time.time()
            board.apply_move(hm[0], hm[1], 2)
            elapsed_ms = (time.time() - t0) * 1000.0
            human_turns += 1
            winner = board.check_winner()
            status = "TERMINAL" if winner else "CONTINUE"
            path = board.get_winning_path(2) if winner == 2 else []
            output_log.append(
                format_human_turn(turn, hm, elapsed_ms,
                                  status, board, winner, path))
            print(f"  T{turn:02d} HUM  -> ({hm[0]},{hm[1]})")

        if winner:
            break
        current_player = 3 - current_player
        turn += 1

    winner = board.check_winner()
    summary = format_summary(winner, turn, ai_turns, human_turns,
                             ai_depths, ai_nodes_list, ai_times)
    output_log.append(summary)
    print(f"\n{board}\n")
    print(summary)
    with open(output_file, 'w', encoding='utf-8') as fh:
        fh.write('\n'.join(output_log))
    print(f"\n[OK] Output -> '{output_file}'")

    avg = lambda lst: sum(lst) / len(lst) if lst else 0
    return {'winner': winner, 'total_turns': turn,
            'ai_turns': ai_turns, 'human_turns': human_turns,
            'avg_depth': avg(ai_depths),
            'avg_nodes': avg(ai_nodes_list),
            'avg_time_ms': avg(ai_times)}

print("Game loop functions defined.")


## 8. Test Case 1 – 7×7 Empty Board

**Setup:** N=7, time_limit=2000ms, completely empty board (matches `inputPS04.txt`).
**Human strategy:** Player B tries to build a left-to-right path along the middle rows.
**Expected outcome:** AI (MCTS) finds a top-to-bottom winning path.


In [ ]:
# Test Case 1: 7x7 empty board, 2000 ms
human_moves_tc1 = [
    (3, 0), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5), (3, 6),
    (2, 0), (2, 1), (2, 2), (4, 0), (4, 1), (1, 0), (5, 0),
    (0, 0), (6, 0), (1, 1), (5, 1), (0, 1), (6, 1),
    (1, 2), (5, 2), (0, 2), (6, 2), (1, 3), (5, 3),
]

stats_tc1 = run_game_test(
    n=7,
    time_limit_ms=2000,
    initial_board=None,
    human_moves=human_moves_tc1,
    output_file=r"C:\Users\Priya_Das\PS04 -Hex Game AI-MCTS\outputPS04_TC1.txt"
)
print(f"\nTest Case 1 Result: winner={stats_tc1['winner']} "
      f"({'AI' if stats_tc1['winner']==1 else 'Human'}), "
      f"turns={stats_tc1['total_turns']}, "
      f"avg_nodes={stats_tc1['avg_nodes']:.0f}")


## 9. Test Case 2 – 9×9 Mid-Game Board

**Setup:** N=9, time_limit=3000ms, partially-filled board.
Player A already has 3 stones along column 4 (rows 0-2).
Player B has a horizontal blocker and one stone at (4,4).
**Tests:** MCTS ability to evaluate a complex mid-game position.


In [ ]:
# Test Case 2: 9x9 mid-game board
n2 = 9
initial_tc2 = [[0]*n2 for _ in range(n2)]

# Player A partial path (top area of col 4)
for r in [0, 1, 2]:
    initial_tc2[r][4] = 1

# Player B horizontal blocker
for c in [2, 3, 5, 6]:
    initial_tc2[3][c] = 2
initial_tc2[4][4] = 2   # centre blocker

# Show initial board
b_init = HexBoard(n2)
for r in range(n2):
    for c in range(n2):
        b_init.board[r][c] = initial_tc2[r][c]
print("TC2 initial board:")
print(b_init)

# Human moves: attempt to close off right edge
human_moves_tc2 = [
    (3, 7), (4, 7), (5, 7), (6, 7), (7, 7), (8, 7),
    (3, 8), (4, 8), (5, 8), (6, 8), (7, 8), (8, 8),
    (2, 8), (1, 8), (0, 8), (2, 7), (1, 7), (0, 7),
    (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1),
]

stats_tc2 = run_game_test(
    n=9,
    time_limit_ms=3000,
    initial_board=initial_tc2,
    human_moves=human_moves_tc2,
    output_file=r"C:\Users\Priya_Das\PS04 -Hex Game AI-MCTS\outputPS04_TC2.txt"
)
print(f"\nTest Case 2 Result: winner={stats_tc2['winner']} "
      f"({'AI' if stats_tc2['winner']==1 else 'Human'}), "
      f"turns={stats_tc2['total_turns']}, "
      f"avg_nodes={stats_tc2['avg_nodes']:.0f}")


## 10. Run Full Pipeline via inputPS04.txt

Runs the complete game using the official `inputPS04.txt` input file.
Uses predefined human moves for automated execution inside Jupyter.

> For real interactive play, run from terminal:
> ```bash
> python PS04_HexGame_MCTS.py inputPS04.txt outputPS04.txt
> ```


In [ ]:
# Full pipeline: read inputPS04.txt and run game
import os

input_path  = r"C:\Users\Priya_Das\PS04 -Hex Game AI-MCTS\inputPS04.txt"
output_path = r"C:\Users\Priya_Das\PS04 -Hex Game AI-MCTS\outputPS04.txt"

# Predefined human moves for automated execution
human_moves_main = [
    (0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6),
    (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0),
    (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1),
    (1, 2), (2, 2), (3, 2), (4, 2), (5, 2), (6, 2),
    (1, 3), (2, 3), (3, 3), (4, 3), (5, 3), (6, 3),
    (1, 4), (2, 4), (3, 4), (4, 4), (5, 4), (6, 4),
    (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (6, 5),
]

n_main, tl_main, board_main = parse_input_file(input_path)
if board_main is not None:
    stats_main = run_game_test(
        n=n_main,
        time_limit_ms=tl_main,
        initial_board=[[board_main.board[r][c] for c in range(n_main)]
                        for r in range(n_main)],
        human_moves=human_moves_main,
        output_file=output_path
    )
    print(f"\nFinal stats: {stats_main}")
    print(f"outputPS04.txt written to: {output_path}")
else:
    print("[ERROR] Could not load inputPS04.txt")


---

## Summary

| Component | Description |
|-----------|-------------|
| **HexBoard** | N×N matrix; 6-adjacency; BFS win detection; path reconstruction |
| **MCTSNode** | UCT formula; two-player win convention (3 − player) |
| **MCTS** | Selection (UCT), Expansion, Simulation (random rollout), Backpropagation |
| **Output** | Exact format: turn banner, stats, winning path, GAME OVER summary |
| **Edge cases** | Invalid moves (re-prompt ×3), out-of-bounds, occupied cells, bad N |

### Submitted files
- `PS04_HexGame_MCTS.py` — single Python file (main implementation)
- `PS04_HexGame_MCTS.ipynb` — this notebook
- `inputPS04.txt` — test input (TC1: 7×7 empty, 2000 ms)
- `outputPS04.txt` — generated output
- `designPS04_[GroupId].pdf` — design document (write separately, ≤4 pages)
- `[GroupId]_Contribution.xlsx` — contribution percentages

### Terminal play
```bash
cd "C:\Users\Priya_Das\PS04 -Hex Game AI-MCTS"
python PS04_HexGame_MCTS.py inputPS04.txt outputPS04.txt
```
